# napari-mp-classifier — demo end-to-end

Clasificación de microplásticos recalcitrantes por **phasores FLIM + espectral**
(Nile Red), contra 6 polímeros de referencia (PET, HDPE, PVC, LDPE, PP, PS).

Este notebook recorre el pipeline completo sobre **datos sintéticos** (la validación
con `.sdt`/`.czi` reales queda pendiente). Cubre las Fases 1–5:
calibración → imagen de muestra → segmentación → features → clasificación → reporte
→ robustez frente al envejecimiento.

Requiere `pip install -e ".[dev]"` desde la raíz del repo.

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '../tests')  # generadores sintéticos

import numpy as np
import matplotlib.pyplot as plt

from napari_mp_classifier import Calibracion, ClasificadorPhasor, analizar_muestra
from napari_mp_classifier.features import extraer_features, matriz_features
from napari_mp_classifier.segmentacion import segmentar
from napari_mp_classifier.metricas import evaluar_clasificacion, evaluar_segmentacion
from napari_mp_classifier.reportes import figura_phasores, figura_segmentacion, generar_reporte
from datos_sinteticos import (
    _columnas, generar_calibracion, generar_particulas,
    generar_imagen_muestra, generar_mascara_celular,
)

## 1. Calibración

Los 6 clusters de referencia en el plano de phasores 4D `[g_flim, s_flim, g_esp, s_esp]`.
En el proyecto real se miden sobre polímero **envejecido con el estándar**
(abrasión + H₂O₂ [+ UV]) — ver `docs/DECISION_CALIBRACION.md`.

In [ ]:
df_cal = generar_calibracion('fusion', n_por_polimero=80, sigma=0.02, semilla=0)
cal = Calibracion.desde_dataframe(df_cal, columnas=_columnas('fusion'))
mediciones = (df_cal[_columnas('fusion')].to_numpy(), df_cal['polimero'].to_numpy())
cal.a_dataframe()

## 2. Imagen de muestra sintética

Campo de microscopía con partículas de los 6 polímeros + materia orgánica + fondo,
y verdad de terreno (segmentación + polímero por partícula).

In [ ]:
canales, verdad = generar_imagen_muestra(semilla=7)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(canales['intensidad'], cmap='magma'); axes[0].set_title('intensidad Nile Red')
axes[1].imshow(canales['g_flim'], cmap='viridis'); axes[1].set_title('g (FLIM) por píxel')
axes[2].imshow(verdad['labels'], cmap='tab20'); axes[2].set_title('verdad de terreno')
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 3. Pipeline completo: `analizar_muestra`

Segmentación → features por ROI → clasificación (+ métricas contra la verdad de terreno).

In [ ]:
res = analizar_muestra(
    canales, cal, estrategia='knn', confianza=0.995,
    mediciones_calibracion=mediciones, escala_um_px=0.18, verdad=verdad,
)
print(res.reporte_segmentacion.resumen())
print()
print(res.reporte_clasificacion.resumen())

In [ ]:
res.conteo_por_polimero()

In [ ]:
res.features.head()

### Figuras del resultado

In [ ]:
fig = figura_segmentacion(
    canales['intensidad'], res.labels,
    etiquetas_por_label={int(l): p for l, p in zip(res.features.index, res.features['polimero_predicho'])},
    titulo='ROIs clasificadas por polímero predicho',
)
plt.show()

In [ ]:
X, columnas = matriz_features(res.features, 'fusion')
fig = figura_phasores(
    cal, X, res.features['polimero_predicho'].to_numpy(), columnas,
    etiquetas_reales=res.features['polimero_real'].to_numpy(), resaltar_errores=True,
    titulo='Phasores de las ROIs sobre los clusters de referencia',
)
plt.show()

### Informe unificado a disco

`generar_reporte` escribe CSV + métricas + figuras + `resumen_muestra.md` en una carpeta.

In [ ]:
rutas = generar_reporte(res, 'salida_notebook', canales=canales, titulo='demo')
sorted(p.name for p in __import__('pathlib').Path('salida_notebook').rglob('*') if p.is_file())

## 4. Robustez frente al envejecimiento

La calibración está en el estándar de envejecimiento; `grado_envejecimiento` mide el
desajuste de la muestra. El modo de falla es conservador: lo que se sale del cluster
va a `no_clasificable`, no a otro polímero.

In [ ]:
clf = ClasificadorPhasor(cal, estrategia='knn', confianza=0.995).entrenar(*mediciones)

grados = np.linspace(-0.2, 0.3, 11)
exactitud, perdido = [], []
for g in grados:
    Xg, yg = generar_particulas('fusion', n_por_polimero=80, n_no_clasificables=0,
                                grado_envejecimiento=g, semilla=5)
    pred = clf.predecir(Xg)
    exactitud.append((pred == yg).mean())
    perdido.append((pred == 'no_clasificable').mean())

plt.figure(figsize=(7, 4))
plt.axvline(0, color='0.7')
plt.plot(grados, exactitud, 'o-', label='exactitud')
plt.plot(grados, perdido, 's--', label='enviado a no_clasificable')
plt.xlabel('desajuste de envejecimiento muestra - calibración')
plt.ylabel('fracción'); plt.legend(); plt.title('Robustez frente al envejecimiento')
plt.show()

### Fusión vs. una sola modalidad bajo envejecimiento

La fusión FLIM + espectral aguanta mejor el desajuste que cualquier modalidad sola.

In [ ]:
filas = []
for m in ('flim', 'espectral', 'fusion'):
    dfm = generar_calibracion(m, n_por_polimero=80, semilla=0)
    calm = Calibracion.desde_dataframe(dfm, columnas=_columnas(m))
    clfm = ClasificadorPhasor(calm, estrategia='knn', confianza=None).entrenar(
        dfm[_columnas(m)].to_numpy(), dfm['polimero'].to_numpy())
    for g in (0.0, 0.1, 0.2):
        Xg, yg = generar_particulas(m, n_por_polimero=100, n_no_clasificables=0,
                                    grado_envejecimiento=g, semilla=8)
        filas.append({'modalidad': m, 'grado': g, 'exactitud': (clfm.predecir(Xg) == yg).mean()})

import pandas as pd
pd.DataFrame(filas).pivot(index='grado', columns='modalidad', values='exactitud')

## 5. Flujo de fagocitos (muestras de monocitos / neutrófilos)

`restringir_a_mascara` conserva solo las partículas dentro de las células que las
fagocitaron (Park et al. 2020).

In [ ]:
from napari_mp_classifier.segmentacion import restringir_a_mascara

canales_c, verdad_c = generar_imagen_muestra(semilla=21)
mascara = generar_mascara_celular(canales_c['intensidad'].shape, verdad_c, fraccion_fagocitada=0.6, semilla=21)
labels = segmentar(canales_c['intensidad'], g_flim=canales_c['g_flim'], s_flim=canales_c['s_flim'])
dentro = restringir_a_mascara(labels, mascara)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(canales_c['intensidad'], cmap='magma'); axes[0].set_title('muestra')
axes[1].imshow(mascara, cmap='gray'); axes[1].set_title('máscara celular')
axes[2].imshow(dentro, cmap='tab20'); axes[2].set_title(f'ROIs fagocitadas ({dentro.max()} de {labels.max()})')
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

---

**Siguientes pasos** (necesitan datos del equipo): `io_crudo.py` para leer
`.sdt`/`.czi` reales, calibración medida sobre el estándar de envejecimiento, y
validación sobre muestras ambientales y de cultivos celulares.